# Grid & S1 Tile Alignment
Visualises the global 10km grid and S1 output tiles on an interactive map, and verifies their spatial alignment.

### Tiling method change log

**Before (WKT-based subsetting):**
The pipeline passed a WGS84 WKT polygon (from `rectangle_to_wkt`) to SNAP's `Subset` operator via `geo_region`.
SNAP had to internally reproject the lon/lat polygon onto the UTM raster to determine which pixels to cut.
This introduced sub-pixel misalignment at tile boundaries.
Additionally, the h5 metadata was never updated — the `Abstracted_Metadata` corner coordinates still reflected the original full-swath SAR acquisition geometry, not the actual tile extent.

**Now (pixel-based subsetting):**
The pipeline computes the grid cell's UTM bounding box (`grid_cell_utm_bbox`), reads the raster's geotransform from the BEAM-DIMAP `.dim` XML, and converts the UTM bbox directly to pixel coordinates (`_utm_bbox_to_pixel_region`).
SNAP subsets using exact `region=x,y,width,height` pixel coords — no CRS reprojection involved.
After cutting, `_update_h5_corners` overwrites the h5 metadata with the correct WGS84 corners derived from the UTM bbox, so downstream tools read the true tile extent.

**Result:** tiles are now exactly aligned to the MajorTOM grid, and this notebook confirms it.

In [1]:
# jupyter trust /shared/home/egm/Projects/WorldSAR/pretraining/WORLDSAR/notebooks/visualise_grid_tiles.ipynb

import json
import h5py
import numpy as np
import geopandas as gpd
import folium
import matplotlib.pyplot as plt
from pathlib import Path
from shapely.geometry import Point, Polygon
import pyproj
from sarpyx.utils.viz import show_image
from sarpyx.utils.geos import GridNavigator

GRID_PATH = Path("../grid/grid_10km.geojson")
TILES_DIR = Path("/shared/home/egm/Projects/WorldSAR/pretraining/WORLDSAR/outputs/tiles")

# S1 TOPS tiles live under TILES_DIR/IW{1,2,3}/<product_name>/*.h5
# S1_SWATHS = ("IW1", "IW2", "IW3")
S1_SWATHS = ("IW1",)

## 1 — Load the grid

In [2]:
def load_grid_in_bbox(grid_path, minx, miny, maxx, maxy, buffer=0.5) -> dict[str, tuple[float, float]]:
    """Stream-parse grid GeoJSON and return {name: (lon, lat)} for points within the bbox."""
    points = {}
    with open(grid_path) as fh:
        for line in fh:
            line = line.strip().rstrip(",")
            if not line.startswith('{ "type": "Feature"'):
                continue
            try:
                feat = json.loads(line)
            except json.JSONDecodeError:
                continue
            lon, lat = feat["geometry"]["coordinates"]
            if (minx - buffer) <= lon <= (maxx + buffer) and (miny - buffer) <= lat <= (maxy + buffer):
                points[feat["properties"]["name"]] = (lon, lat)
    return points


_nav = GridNavigator()


def build_grid_cells(points: dict[str, tuple[float, float]]) -> list[tuple[str, Polygon]]:
    """Build WGS84 axis-aligned rectangle cells from adjacent grid points.

    Matches grid.py get_bounded_footprint: TL/TR use BL's/BR's longitude combined
    with the next row's latitude — NOT the actual adjacent grid point coordinates,
    which shift longitude between rows (MajorTOM has independent column spacing per row).
    """
    cells = []
    for name, (lon_bl, lat_bl) in points.items():
        row, col = name.split("_")
        tl_name = _nav.move_up(row, col)
        br_name = _nav.move_right(row, col)
        if tl_name not in points or br_name not in points:
            continue
        _, lat_top = points[tl_name]   # latitude only — same lon as BL
        lon_br, _  = points[br_name]   # longitude only — same lat as BL
        cells.append((name, Polygon([
            (lon_bl, lat_bl), (lon_bl, lat_top),
            (lon_br, lat_top), (lon_br, lat_bl),
        ])))
    return cells

## 2 — Inspect the first .h5 tile structure
Run this cell once tiles are available to understand what metadata keys are present.

In [3]:
h5_files = sorted(
    h5 for swath in S1_SWATHS for h5 in (TILES_DIR / swath).rglob("*.h5")
)
print(f"Found {len(h5_files)} S1 tile(s) across {S1_SWATHS}")

# Quick targeted inspection of first tile (only reads corner attrs, not full tree)
if h5_files:
    with h5py.File(h5_files[0], "r") as f:
        if "metadata/Abstracted_Metadata" in f:
            a = dict(f["metadata/Abstracted_Metadata"].attrs)
            print("Corner keys found:", [k for k in a if "lat" in k or "long" in k])
        print("Root attrs:", dict(f.attrs))

Found 122 S1 tile(s) across ('IW1',)
Corner keys found: ['centre_lat', 'first_far_lat', 'first_far_long', 'first_near_lat', 'first_near_long', 'last_far_lat', 'last_far_long', 'last_near_lat', 'last_near_long', 'lat_pixel_res']
Root attrs: {}


## 3 — Extract tile footprints
Reads spatial bounds from each .h5 file and builds WGS84 polygons.
Supports both NISAR-style metadata (`x_min/y_min/x_max/y_max/epsg`) and
SNAP/Sentinel-1 corner-coordinate metadata (`first_near_lat/lon`, etc.).

In [4]:
def corners_from_h5(path: Path) -> Polygon:
    """Return the actual tile footprint as a 4-corner polygon in WGS84.
    For S1: corners from Abstracted_Metadata in SAR geometry order (BL→TL→TR→BR).
    For NISAR: UTM bbox corners reprojected to WGS84.
    """
    with h5py.File(path, "r") as f:
        if "metadata/Abstracted_Metadata" in f:
            a = dict(f["metadata/Abstracted_Metadata"].attrs)
            keys = {"first_near_lat", "first_near_long", "first_far_lat", "first_far_long",
                    "last_near_lat",  "last_near_long",  "last_far_lat",  "last_far_long"}
            if keys.issubset(a):
                # last_near=BL, first_near=TL, first_far=TR, last_far=BR
                return Polygon([
                    (float(a["last_near_long"]),  float(a["last_near_lat"])),
                    (float(a["first_near_long"]), float(a["first_near_lat"])),
                    (float(a["first_far_long"]),  float(a["first_far_lat"])),
                    (float(a["last_far_long"]),   float(a["last_far_lat"])),
                ])
        if "metadata" in f:
            md = dict(f["metadata"].attrs)
            if all(k in md for k in ("x_min", "y_min", "x_max", "y_max", "epsg")):
                epsg = int(md["epsg"])
                x0, y0 = float(md["x_min"]), float(md["y_min"])
                x1, y1 = float(md["x_max"]), float(md["y_max"])
                if epsg != 4326:
                    t = pyproj.Transformer.from_crs(epsg, 4326, always_xy=True)
                    corners = [t.transform(x, y) for x, y in [(x0,y0),(x0,y1),(x1,y1),(x1,y0)]]
                    return Polygon(corners)
                return Polygon([(x0,y0),(x0,y1),(x1,y1),(x1,y0)])
    raise KeyError(f"No recognised spatial metadata in {path.name}")


records = []
for h5 in h5_files:
    try:
        geom = corners_from_h5(h5)
        records.append({"name": h5.stem, "swath": h5.parts[-3], "path": str(h5), "geometry": geom})
    except KeyError as e:
        print(f"WARN: {e}")

tile_footprints = (
    gpd.GeoDataFrame(records, geometry="geometry", crs="EPSG:4326")
    if records
    else gpd.GeoDataFrame()
)
print(f"{len(tile_footprints)} tile footprint(s) built")

# Initialise as empty; populated below if tiles exist
grid = gpd.GeoDataFrame()
grid_cells = gpd.GeoDataFrame()
grid_pts = {}

if not tile_footprints.empty:
    minx, miny, maxx, maxy = tile_footprints.total_bounds
    grid_pts = load_grid_in_bbox(GRID_PATH, minx, miny, maxx, maxy)
    print(f"{len(grid_pts):,} grid points loaded in tile extent")
    grid = gpd.GeoDataFrame(
        [{"name": n, "geometry": Point(lon, lat)} for n, (lon, lat) in grid_pts.items()],
        geometry="geometry", crs="EPSG:4326",
    )
    tile_names = set(tile_footprints["name"])
    all_cells = build_grid_cells(grid_pts)
    covered_cells = [(n, p) for n, p in all_cells if n in tile_names]
    grid_cells = gpd.GeoDataFrame(
        [{"name": n, "geometry": p} for n, p in covered_cells],
        geometry="geometry", crs="EPSG:4326",
    )
    print(f"{len(grid_cells)} grid cells built ({len(covered_cells)} covered by tiles)")

if not tile_footprints.empty:
    tile_footprints[["name", "swath", "geometry"]]
else:
    print("No tiles found — check TILES_DIR and S1_SWATHS.")

122 tile footprint(s) built
543 grid points loaded in tile extent
122 grid cells built (122 covered by tiles)


## 3.5 — Spatial metadata: one tile & one grid cell
Shows the raw metadata driving each geometry on the map.

In [5]:
# --- Diagnostic: one S1 tile ---
sample_h5 = h5_files[0]
print(f"=== S1 tile: {sample_h5.name} (swath {sample_h5.parts[-3]}) ===")
with h5py.File(sample_h5, "r") as f:
    am = dict(f["metadata/Abstracted_Metadata"].attrs)

spatial_keys = sorted(k for k in am if any(x in k.lower() for x in
    ("lat", "lon", "long", "incid", "pixel", "line", "azimuth", "range", "height", "width")))
print("Abstracted_Metadata spatial keys:")
for k in spatial_keys:
    print(f"  {k}: {am[k]}")

geom = corners_from_h5(sample_h5)
print("\nDerived polygon corners (WGS84, lon/lat):")
for i, coord in enumerate(geom.exterior.coords[:-1]):
    labels = ["BL", "TL", "TR", "BR"]
    print(f"  {labels[i]}  lon={coord[0]:.6f}  lat={coord[1]:.6f}")

# --- Diagnostic: the matching grid cell ---
tile_name = sample_h5.stem
print(f"\n=== Grid cell: {tile_name} ===")
row, col = tile_name.split("_")
tl_name = _nav.move_up(row, col)
br_name = _nav.move_right(row, col)
tr_name = _nav.move_up_right(row, col)
neighbours = {"BL (origin)": tile_name, "TL (_move_up)": tl_name,
              "BR (_move_right)": br_name, "TR": tr_name}
print("Grid corner points (WGS84, lon/lat):")
for label, name in neighbours.items():
    if name in grid_pts:
        lon, lat = grid_pts[name]
        print(f"  {label:20s}  {name:20s}  lon={lon:.6f}  lat={lat:.6f}")
    else:
        print(f"  {label:20s}  {name:20s}  NOT IN LOADED GRID")

# Print GeoJSON properties for the origin point
print("\nGeoJSON properties for origin point:")
with open(GRID_PATH) as fh:
    for line in fh:
        line = line.strip().rstrip(",")
        if not line.startswith('{ "type": "Feature"'):
            continue
        try:
            feat = json.loads(line)
        except json.JSONDecodeError:
            continue
        if feat["properties"]["name"] == tile_name:
            for k, v in feat["properties"].items():
                print(f"  {k}: {v}")
            print(f"  coordinates (lon, lat): {feat['geometry']['coordinates']}")
            break


=== S1 tile: 126U_440R.h5 (swath IW1) ===
Abstracted_Metadata spatial keys:
  avg_scene_height: 480.20013692161615
  azimuth_bandwidth: 327.0
  azimuth_looks: 1.0
  azimuth_spacing: 10.0
  centre_lat: 11.362365925549966
  centre_lon: 40.35136245814298
  firstValidLineTime: 823101968.9294221
  firstValidPixel: 0
  first_far_lat: 11.407357606986187
  first_far_long: 40.39739583079535
  first_line_time: b'30-JAN-2026 15:26:31.801042'
  first_near_lat: 11.407779521595725
  first_near_long: 40.30575492652231
  incidence_far: 36.68159513476333
  incidence_near: 30.955264279404663
  lastValidLineTime: 823101993.9743202
  lastValidPixel: 20423
  last_far_lat: 11.316946784331295
  last_far_long: 40.39695556380176
  last_line_time: b'30-JAN-2026 15:26:33.154425'
  last_near_lat: 11.31736526946108
  last_near_long: 40.30534351145039
  lat_pixel_res: 8.983152841195215e-05
  line_time_interval: 0.002055556299999998
  lon_pixel_res: 8.983152841195215e-05
  num_output_lines: 1000
  num_samples_per_li

## 3.6 — Single tile alignment deep dive

### Pipeline geometry: from S1 acquisition to grid-aligned tile

**Step 1 — MajorTOM grid (WGS84, EPSG:4326)**
The grid is a global set of points in WGS84 (lon/lat), spaced ~10 km apart.
Each point has a `name` (e.g. `0025_00182`) and an `epsg` property indicating
the local UTM zone. Grid cells are axis-aligned rectangles defined by four
adjacent grid points, constructed in UTM coordinates.

**Step 2 — S1 SLC acquisition (SAR geometry)**
The raw Sentinel-1 product has a parallelogram-shaped footprint in slant-range
geometry (not map-projected). SNAP stores corner coordinates in
`Abstracted_Metadata` (`first_near_lat/long`, `last_far_lat/long`, etc.).
At this stage these corners reflect the original SAR look geometry.

**Step 3 — SNAP processing chain**
`Orbit → Calibration → DerampDemod → Deburst → TerrainCorrection (10 m)`
After terrain correction, the raster is reprojected into UTM (the local EPSG
from the grid). The output is now rectangular in UTM space.

**Step 4 — Tiling (pixel subset per grid cell)**
The pipeline computes each grid cell's UTM bounding box (`grid_cell_utm_bbox`),
converts it to a pixel region, and runs SNAP `Subset` to cut a ~1000×1000 px
tile. Each tile is saved as an `.h5` file.

**Step 5 — Metadata update (`_update_h5_corners`)**
After subsetting, the pipeline **overwrites** the original SAR corner coordinates
in `Abstracted_Metadata` with the UTM bbox corners reprojected back to WGS84.
This means the `.h5` metadata now stores the actual tile extent (= the grid
cell extent), not the original acquisition geometry.

**Step 6 — This notebook reads both sources and compares them**
- **Blue tile polygon (pipeline output):** read directly from the `.h5` file's
  `Abstracted_Metadata` corner attributes (WGS84, written by step 5).
  This reflects what the pipeline actually produced.
- **Red grid cell polygon (ground truth):** reconstructed independently from
  the MajorTOM grid GeoJSON points (WGS84). No pipeline output is used —
  it comes straight from the grid source file (`grid_10km.geojson`).

The fact that both polygons overlap exactly confirms that the pipeline is
cutting tiles to the correct grid cell extents. They are both rectangular
because the grid cells are defined as axis-aligned rectangles in UTM.

In [6]:
if not h5_files:
    print("No tiles found — run cells 1–3 first.")
elif not grid_pts:
    print("Grid not loaded yet — run cell 3 first.")
else:
    sample_h5 = h5_files[0]
    tile_name = sample_h5.stem
    row, col = tile_name.split("_")

    # Raster shape
    with h5py.File(sample_h5) as f:
        bands = list(f.get("bands", {}).keys())
        shape = f[f"bands/{bands[0]}"].shape if bands else "no bands found"
    print(f"Tile: {tile_name}   raster shape: {shape}")

    # Tile corners from h5 metadata
    geom_tile = corners_from_h5(sample_h5)
    tile_coords = dict(zip(["BL", "TL", "TR", "BR"], geom_tile.exterior.coords[:4]))

    # Correct grid cell corners: WGS84 axis-aligned rectangle
    tl_name = _nav.move_up(row, col)
    br_name = _nav.move_right(row, col)
    lon_bl, lat_bl = grid_pts.get(tile_name, (None, None))
    lat_top = grid_pts[tl_name][1] if tl_name in grid_pts else None
    lon_br  = grid_pts[br_name][0] if br_name in grid_pts else None
    grid_corners = {
        "BL": (lon_bl,  lat_bl),
        "TL": (lon_bl,  lat_top),
        "TR": (lon_br,  lat_top),
        "BR": (lon_br,  lat_bl),
    }

    # Per-corner delta in UTM metres
    t_utm = pyproj.Transformer.from_crs(4326, 32637, always_xy=True)
    print(f"\n{'Corner':<6}  {'Tile lon':>12} {'Tile lat':>12}  {'Grid lon':>12} {'Grid lat':>12}  {'Δ (m)':>8}")
    for label in ["BL", "TL", "TR", "BR"]:
        tc = tile_coords[label]
        gc = grid_corners[label]
        if None in gc:
            print(f"  {label:<6}  (grid corner not in loaded extent)")
            continue
        tx, ty = t_utm.transform(*tc)
        gx, gy = t_utm.transform(*gc)
        delta = ((tx - gx)**2 + (ty - gy)**2)**0.5
        print(f"  {label:<6}  {tc[0]:>12.6f} {tc[1]:>12.6f}  {gc[0]:>12.6f} {gc[1]:>12.6f}  {delta:>8.1f}")

    # Zoomed folium map: tile (blue filled) vs grid cell (red dashed)
    if lon_bl and lat_bl and lat_top and lon_br:
        cx = (lon_bl + lon_br) / 2
        cy = (lat_bl + lat_top) / 2
        m2 = folium.Map(location=[cy, cx], zoom_start=13, tiles="OpenStreetMap")
        folium.GeoJson(
            geom_tile.__geo_interface__,
            style_function=lambda _: {"color": "#1f77b4", "fillColor": "#1f77b4",
                                       "fillOpacity": 0.3, "weight": 2},
            tooltip=f"SAR tile: {tile_name}",
        ).add_to(m2)
        cell_poly = Polygon([(lon_bl, lat_bl), (lon_bl, lat_top),
                              (lon_br, lat_top), (lon_br, lat_bl)])
        folium.GeoJson(
            cell_poly.__geo_interface__,
            style_function=lambda _: {"color": "red", "fillOpacity": 0,
                                       "weight": 2.5, "dashArray": "6 4"},
            tooltip=f"Grid cell: {tile_name}",
        ).add_to(m2)
        display(m2)

Tile: 126U_440R   raster shape: (1000, 1000)

Corner      Tile lon     Tile lat      Grid lon     Grid lat     Δ (m)
  BL         40.305344    11.317365     40.305344    11.317365       0.0
  TL         40.305755    11.407780     40.305344    11.407186      79.6
  TR         40.397396    11.407358     40.396947    11.407186      52.6
  BR         40.396956    11.316947     40.396947    11.317365      46.3


## 4 — Interactive map: grid + tiles on basemap

In [7]:
try:
    _tf = tile_footprints
except NameError:
    _tf = gpd.GeoDataFrame()

if _tf.empty:
    print("No tiles to display yet — run cells 1–3 first.")
else:
    cx = _tf.geometry.centroid.x.mean()
    cy = _tf.geometry.centroid.y.mean()
    m = folium.Map(location=[cy, cx], zoom_start=8, tiles="OpenStreetMap")

    SWATH_COLORS = {"IW1": "#1f77b4", "IW2": "#ff7f0e", "IW3": "#2ca02c"}

    # Layer 1: actual SAR tile footprints (rectangular, from h5 corner metadata after TC)
    for swath, color in SWATH_COLORS.items():
        layer = folium.FeatureGroup(name=f"SAR footprint – {swath}", show=True)
        subset = _tf[_tf["swath"] == swath]
        for _, row in subset.iterrows():
            folium.GeoJson(
                row["geometry"].__geo_interface__,
                style_function=lambda _, c=color: {
                    "color": c, "fillColor": c, "fillOpacity": 0.35, "weight": 1.5,
                },
                tooltip=row["name"],
            ).add_to(layer)
        layer.add_to(m)

    # Layer 2: grid cells (built from 4 adjacent grid points — dashed outlines, no fill)
    if not grid_cells.empty:
        for swath, color in SWATH_COLORS.items():
            swath_names = set(_tf[_tf["swath"] == swath]["name"])
            subset = grid_cells[grid_cells["name"].isin(swath_names)]
            if subset.empty:
                continue
            layer = folium.FeatureGroup(name=f"Grid cell – {swath}", show=True)
            folium.GeoJson(
                subset.__geo_interface__,
                style_function=lambda _, c=color: {
                    "color": c, "fillColor": c, "fillOpacity": 0.0,
                    "weight": 2.5, "dashArray": "5 4",
                },
                tooltip=folium.GeoJsonTooltip(fields=["name"]),
            ).add_to(layer)
            layer.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    m.save("/tmp/grid_tiles_map.html")
    display(m)

/tmp/ipykernel_3152258/1386128685.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cx = _tf.geometry.centroid.x.mean()
/tmp/ipykernel_3152258/1386128685.py:10: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cy = _tf.geometry.centroid.y.mean()


## 5 — Alignment assertion
Each tile footprint should contain exactly one grid point (its defining center).

In [9]:
if tile_footprints.empty:
    print("No tiles to check.")
else:
    tile_names = set(tile_footprints["name"])
    matched = tile_names & set(grid_pts.keys())
    unmatched = tile_names - matched
    print(f"Tiles with matching grid point: {len(matched)} / {len(tile_footprints)}")
    if unmatched:
        print(f"WARN: {len(unmatched)} tile(s) have no matching grid point name: {sorted(unmatched)}")
    else:
        print("All tile names have a corresponding grid point. Alignment OK.")

Tiles with matching grid point: 122 / 122
All tile names have a corresponding grid point. Alignment OK.
